In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Etapele Transpiler-ului

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.4.0
```
</details>
Această pagină descrie etapele pipeline-ului de transpilare preconfigurate din Qiskit SDK. Există șase etape:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

Funcția [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) creează un [staged pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) preconfigurate, compus din aceste etape. Pasele specifice care alcătuiesc fiecare etapă depind de argumentele transmise lui `generate_preset_pass_manager`. `optimization_level` este un argument pozițional care trebuie specificat; este un număr întreg care poate fi 0, 1, 2 sau 3. Valorile mai mari indică o optimizare mai intensă, dar și mai costisitoare (vezi [Valorile implicite ale Transpiler-ului și opțiunile de configurare](defaults-and-configuration-options)).

Modalitatea recomandată de a transpila un circuit este să creezi un staged pass manager preconfigurate și apoi să rulezi acel pass manager pe Circuit, după cum este descris în [Transpilare cu pass managere](transpile-with-pass-managers). Totuși, o alternativă mai simplă, dar mai puțin personalizabilă, este să folosești funcția [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). Această funcție acceptă direct Circuit-ul ca argument. La fel ca în cazul `generate_preset_pass_manager`, pasele specifice de transpilare utilizate depind de argumentele, precum `optimization_level`, transmise lui `transpile`. De fapt, intern, funcția `transpile` apelează `generate_preset_pass_manager` pentru a crea un staged pass manager preconfigurate și îl rulează pe Circuit.
## Etapa Init
Această primă etapă face foarte puțin în mod implicit și este utilă în principal dacă vrei să incluzi propriile tale optimizări inițiale. Deoarece majoritatea algoritmilor de layout și routing sunt proiectați să lucreze doar cu Gate-uri pe unul sau doi qubiți, această etapă este folosită și pentru a traduce orice Gate-uri care operează pe mai mult de doi Qubiți, în Gate-uri care operează doar pe unul sau doi Qubiți.

Pentru mai multe informații despre implementarea propriilor optimizări inițiale pentru această etapă, consultă secțiunea despre plugin-uri și personalizarea pass managerelor.
## Etapa Layout
Următoarea etapă implică layout-ul sau conectivitatea Backend-ului căruia îi va fi trimis un circuit. În general, circuitele cuantice sunt entități abstracte ale căror Qubiți sunt reprezentări „virtuale" sau „logice" ale Qubiților reali utilizați în calcule. Pentru a executa o secvență de poartă-uri, este necesară o mapare unu-la-unu de la Qubiții „virtuali" la Qubiții „fizici" dintr-un dispozitiv cuantic real. Această mapare este stocată ca un obiect `Layout` și face parte din constrângerile definite în cadrul [arhitecturii setului de instrucțiuni (ISA)](/guides/transpile#instruction-set-architecture) a unui Backend.

![Această imagine ilustrează maparea Qubiților din reprezentarea de tip fir către o diagramă care reprezintă modul în care Qubiții sunt conectați pe QPU.](../docs/images/guides/transpiler-stages/layout-mapping.svg "Maparea Qubiților")

Alegerea mapării este extrem de importantă pentru minimizarea numărului de operații SWAP necesare pentru a mapa Circuit-ul de intrare pe topologia dispozitivului și pentru a asigura utilizarea celor mai bine calibrați Qubiți. Datorită importanței acestei etape, pass managerele preconfigurate încearcă câteva metode diferite pentru a găsi cel mai bun layout. De obicei, acest lucru implică doi pași: mai întâi, se încearcă găsirea unui layout „perfect" (un layout care nu necesită operații SWAP), iar apoi, un pas euristic care încearcă să găsească cel mai bun layout de utilizat dacă un layout perfect nu poate fi găsit. Există două `Pase` utilizate de obicei pentru acest prim pas:

- `TrivialLayout`: Mapează naiv fiecare qubit virtual la același qubit fizic numerotat pe dispozitiv (adică, [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). Acesta este comportamentul istoric folosit doar în `optimzation_level=1` pentru a încerca să găsească un layout perfect. Dacă eșuează, `VF2Layout` este încercat în continuare.
- `VF2Layout`: Acesta este un `AnalysisPass` care selectează un layout ideal tratând această etapă ca o problemă de izomorfism de subgraf, rezolvată de algoritmul VF2++. Dacă se găsesc mai multe layout-uri, se rulează o euristică de punctare pentru a selecta maparea cu cea mai mică eroare medie.

Apoi, pentru etapa euristică, două pase sunt utilizate în mod implicit:

- `DenseLayout`: Găsește sub-graful dispozitivului cu cea mai mare conectivitate și care are același număr de Qubiți ca și Circuit-ul (utilizat pentru nivelul de optimizare 1 dacă există operații de flux de control (cum ar fi IfElseOp) prezente în Circuit).
- `SabreLayout`: Această pasă selectează un layout pornind de la un layout inițial aleatoriu și rulând în mod repetat algoritmul `SabreSwap`. Această pasă este folosită doar la nivelurile de optimizare 1, 2 și 3, dacă un layout perfect nu este găsit prin pasa `VF2Layout`. Pentru mai multe detalii despre acest algoritm, consultă lucrarea [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)
## Etapa Routing
Pentru a implementa un poartă cu doi Qubiți între Qubiți care nu sunt conectați direct pe un dispozitiv cuantic, unul sau mai multe Gate-uri SWAP trebuie inserate în Circuit pentru a muta stările Qubiților până când acestea sunt adiacente pe harta Gate-urilor dispozitivului. Fiecare poartă SWAP reprezintă o operație costisitoare și zgomotoasă de efectuat. Astfel, găsirea numărului minim de poartă-uri SWAP necesare pentru a mapa un circuit pe un dispozitiv dat este un pas important în procesul de transpilare. Din motive de eficiență, această etapă este de obicei calculată împreună cu etapa Layout în mod implicit, dar ele sunt distincte logic una față de cealaltă. Etapa *Layout* selectează Qubiții hardware care urmează să fie utilizați, în timp ce etapa *Routing* inserează numărul adecvat de poartă-uri SWAP pentru a executa circuitele folosind layout-ul selectat.

Totuși, găsirea mapării SWAP optime este dificilă. De fapt, este o problemă NP-hard și, prin urmare, este prohibitiv de costisitoare de calculat pentru toate dispozitivele cuantice, cu excepția celor mai mici, și a circuitelor de intrare. Pentru a depăși acest lucru, Qiskit folosește un algoritm euristic stocastic numit `SabreSwap` pentru a calcula o mapare SWAP bună, dar nu neapărat optimă. Utilizarea unei metode stochastice înseamnă că circuitele generate nu sunt garantate să fie aceleași la rulări repetate. Într-adevăr, rularea aceluiași Circuit în mod repetat duce la o distribuție a adâncimilor circuitului și a numărului de poartă-uri la ieșire. Din acest motiv, mulți utilizatori aleg să ruleze funcția de routing (sau întregul `StagedPassManager`) de mai multe ori și să selecteze circuitele cu cea mai mică adâncime din distribuția ieșirilor.

De exemplu, să luăm un circuit GHZ de 15 Qubiți executat de 100 de ori, folosind un `initial_layout` „prost" (deconectat).

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.fake_provider import GenericBackendV2

backend = GenericBackendV2(15)


ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/62479697-cef1-4a89-ac2a-20051dd294f4-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ab89c4ea-06c4-4320-b493-feb691b3570d-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/f6a3a92a-8656-4518-ba2c-c3b0b038f507-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](/docs/guides/transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/d4d1f65a-3336-4d70-9189-65ba010f2366-1.svg" alt="Output of the previous code cell" />

![Ieșirea celulei de cod anterioare](../docs/images/guides/transpiler-stages/extracted-outputs/f6a3a92a-8656-4518-ba2c-c3b0b038f507-0.svg)

După cum poți vedea, acest circuit trebuie să execute un poartă cu doi Qubiți între Qubiții 0 și 14, care sunt foarte depărtați în graful de conectivitate. Rularea acestui Circuit necesită astfel inserarea de poartă-uri SWAP pentru a executa toate Gate-urile cu doi Qubiți folosind pasa `SabreSwap`.

Notează, de asemenea, că algoritmul `SabreSwap` este diferit de metoda mai mare `SabreLayout` din etapa anterioară. În mod implicit, `SabreLayout` rulează atât layout-ul cât și routing-ul, și returnează Circuit-ul transformat. Acest lucru se face din câteva motive tehnice particulare specificate pe [pagina de referință API](../api/qiskit/qiskit.transpiler.passes.SabreLayout) a pasei.
## Etapa de traducere
Când scrii un circuit cuantic, ești liber să folosești orice poartă cuantică (operație unitară) dorești, împreună cu o colecție de operații care nu sunt porți, cum ar fi instrucțiunile de măsurare sau resetare a qubiților. Totuși, majoritatea dispozitivelor cuantice acceptă nativ doar un număr mic de operații Gate cuantice și non-Gate. Aceste poartă-uri native fac parte din definiția ISA a unei ținte ([ISA](/guides/transpile#instruction-set-architecture)), iar această etapă a `PassManagers`-elor preset traduce (sau *desfășoară*) Gate-urile specificate într-un Circuit în Gate-urile de bază native ale unui Backend specificat. Aceasta este un pas important, deoarece permite executarea circuitului de către Backend, dar în general duce la o creștere a adâncimii și a numărului de poartă-uri.

Două cazuri speciale sunt deosebit de importante de subliniat și ajută la ilustrarea a ceea ce face această etapă.

1. Dacă un poartă SWAP nu este un poartă nativ al Backend-ului țintă, acesta necesită trei Gate-uri CNOT:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4552b367-75e9-4faa-a59d-8317e25ac145-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2700405a-f559-45d3-99a9-5b4447621743-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/d4d1f65a-3336-4d70-9189-65ba010f2366-1.svg)

Ca produs a trei Gate-uri CNOT, un SWAP este o operație costisitoare de efectuat pe dispozitivele cuantice cu zgomot. Cu toate acestea, astfel de operații sunt de obicei necesare pentru a încorpora un circuit în conectivitățile limitate de poartă ale multor dispozitive. Prin urmare, minimizarea numărului de poartă-uri SWAP dintr-un Circuit este un obiectiv principal în procesul de transpilare.

2. Un poartă Toffoli, sau controlled-controlled-not (`ccx`), este un poartă cu trei qubiți. Deoarece setul nostru de poartă-uri de bază include doar Gate-uri cu unul sau doi qubiți, această operație trebuie descompusă. Cu toate acestea, este destul de costisitoare:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = GenericBackendV2(5)

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ae2d9390-5c26-46f3-9418-a684ba8a406a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/4552b367-75e9-4faa-a59d-8317e25ac145-0.svg)

Pentru fiecare poartă Toffoli dintr-un Circuit cuantic, hardware-ul poate executa până la șase Gate-uri CNOT și câteva Gate-uri cu un singur qubit. Acest exemplu demonstrează că orice algoritm care utilizează mai multe Gate-uri Toffoli va rezulta într-un Circuit cu adâncime mare și va fi, prin urmare, afectat considerabil de zgomot.
## Etapa de optimizare
Această etapă se concentrează pe descompunerea circuitelor cuantice în setul de poartă-uri de bază al dispozitivului țintă și trebuie să combată creșterea adâncimii din etapele de layout și rutare. Din fericire, există multe rutine pentru optimizarea circuitelor fie prin combinarea, fie prin eliminarea Gate-urilor. În unele cazuri, aceste metode sunt atât de eficiente încât circuitele de ieșire au adâncime mai mică decât cele de intrare, chiar și după maparea pe topologia hardware prin layout și rutare. În alte cazuri, nu se poate face prea mult, iar calculul poate fi dificil de efectuat pe dispozitivele cu zgomot. Această etapă este locul unde diferitele niveluri de optimizare încep să difere.

- Pentru `optimization_level=1`, această etapă pregătește [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) și [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), care combină lanțuri de poartă-uri cu un singur qubit și anulează orice Gate-uri CNOT consecutive.
- Pentru `optimization_level=2`, această etapă folosește pasul [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) în locul `CXCancellation`, care elimină Gate-urile redundante prin exploatarea relațiilor de comutativitate.
- Pentru `optimization_level=3`, această etapă pregătește următorii pași:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

În plus, această etapă execută și câteva verificări finale pentru a se asigura că toate instrucțiunile din Circuit sunt compuse din Gate-urile de bază disponibile pe Backend-ul țintă.

Exemplul de mai jos, folosind o stare GHZ, demonstrează efectele diferitelor setări ale nivelului de optimizare asupra adâncimii circuitului și a numărului de poartă-uri.

> **Note:** Ieșirea transpilării variază din cauza mapperului SWAP stocastic. Prin urmare, numerele de mai jos se vor schimba cel mai probabil de fiecare dată când rulezi codul.

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

Următorul cod construiește o stare GHZ cu 15 qubiți și compară `optimization_levels` ale transpilării în termeni de adâncime a circuitului rezultat, număr de poartă-uri și număr de poartă-uri cu mai mulți qubiți.